In [20]:
from env import config
import requests
import json

In [21]:
FLIGHT_PRICE_PREDICTOR_MODEL = config("FLIGHT_PRICE_PREDICTOR_MODEL", default = "flight_price_predictor")
to_train = "flight_price_predictor_a"
if FLIGHT_PRICE_PREDICTOR_MODEL == to_train:
    to_train = "flight_price_predictor_b"

In [22]:
print("Training", to_train)

Training flight_price_predictor_b


In [23]:
sql_command1 = f"""DROP MODEL {to_train};"""

sql_command2 = f"""
CREATE MODEL mindsdb.{to_train}
FROM ai_travel_agent (
    SELECT * FROM flight_prices
    )
PREDICT totalFare
ORDER BY flightDate
GROUP BY startingAirport, destinationAirport, isBasicEconomy, isRefundable, isNonStop, segmentsAirlineName
WINDOW 30
HORIZON 10;

DESCRIBE PREDICTOR mindsdb.flight_price_predictor_a;
"""
commands_to_run = [sql_command1, sql_command2]

In [24]:
def mindsdb_query(sql_query):
    url = 'http://127.0.0.1:47334/api/sql/query'
    headers = {"Content-Type": "application/json"}
    return requests.post(url, json={'query':sql_query}, headers=headers)

In [25]:
for cmd in commands_to_run:
    mindsdb_query(cmd)

In [31]:
sql_command_3 = f"DESCRIBE PREDICTOR mindsdb.{to_train};"
response = mindsdb_query(sql_command_3)
response

<Response [200]>

In [29]:
response.json()

{'type': 'error',
 'error_code': 0,
 'error_message': 'Model not found: mindsdb.flight_price_predictor_b',
 'context': {'show_secrets': False, 'db': 'mindsdb'}}

In [28]:
response_data = response.json()
column_names = response_data['column_names']
# print(column_names)
dataset = response_data['data'][0]
# print(dataset)
for i, col in enumerate(column_names):
    print(col, dataset[i])

KeyError: 'column_names'